# Notebook 4: Model Context Protocol (MCP) -- Connecting Agents to External Tool Servers

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the **Model Context Protocol (MCP)** and its client-server architecture.
2. Build an **MCP server** using FastMCP that exposes tools over SSE.
3. Connect a **LangGraph agent** to an MCP server using `langchain-mcp-adapters` (the easy way).
4. Connect a **MAF agent** to an MCP server by manually constructing Python function signatures (the manual way).
5. Understand the **trade-offs** between adapter-based and manual MCP integration.

---

## Why MCP Matters

In every notebook so far, we have defined tools **inside** the agent code -- a `@tool` decorated function, a hardcoded API call, etc. This works fine for small projects, but it creates tight coupling: every time you add, remove, or update a tool, you must redeploy the agent.

**MCP decouples tool development from agent development.** A separate server advertises its tools dynamically, and any compatible agent can discover and call them at runtime.

```
+--------------+         SSE / HTTP          +-------------------+
|   Agent      | <-------------------------> |   MCP Server      |
|  (Client)    |   1. List available tools    |  (Tool Provider)  |
|              |   2. Call tool by name        |                   |
|  LangGraph   |   3. Receive result           |  get_weather()    |
|  or MAF      |                               |  get_time()       |
+--------------+                               +-------------------+
```

**Key benefits:**
- Update tools without redeploying agents.
- Multiple agents can share the same tool server.
- Teams can develop tools and agents independently.
- Tool discovery is dynamic -- agents adapt to whatever the server offers.

## Section 1: The MCP Server

Our workshop includes a ready-made MCP server at `utility/mcp-server.py`. It uses **FastMCP** to expose two tools over Server-Sent Events (SSE).

Here is the full server code (for reference -- do not run this cell, as the server blocks the kernel):

```python
from mcp.server.fastmcp import FastMCP
import datetime
from zoneinfo import ZoneInfo

mcp = FastMCP(
    name="langGraph_Demo",
    host="0.0.0.0",
    port="8787",
)

@mcp.tool()
def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city."""
    if city.lower() == "new york":
        return {
            "status": "success",
            "report": "The weather in New York is sunny with a temperature of 25 degrees Celsius.",
        }
    else:
        return {"status": "error", "error_message": f"Weather for '{city}' is not available."}

@mcp.tool()
def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city."""
    if city.lower() == "new york":
        tz = ZoneInfo("America/New_York")
        now = datetime.datetime.now(tz)
        return {"status": "success", "report": f"Current time in {city}: {now.strftime('%H:%M:%S %Z')}"}
    else:
        return {"status": "error", "error_message": f"No timezone info for {city}."}

if __name__ == "__main__":
    mcp.run(transport="sse")
```

**What to notice:**
- `@mcp.tool()` is the only decorator needed. FastMCP reads the function signature and docstring to build the JSON schema automatically.
- `transport="sse"` means the server uses Server-Sent Events, listening on port 8787.
- The tools are intentionally simple (hardcoded responses for New York only) so we can focus on the integration pattern.

## Section 2: Starting the MCP Server

We launch the MCP server as a **background subprocess** so the notebook kernel stays free for our agent code.

In [1]:
import subprocess
import sys
import time
import os

# Path to the MCP server script
server_script = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "utility", "mcp-server.py"))
print(f"Server script: {server_script}")
print(f"Exists: {os.path.exists(server_script)}")

# Start the server in the background
server_proc = subprocess.Popen(
    [sys.executable, server_script],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # Give the server time to start
print(f"MCP server started (PID: {server_proc.pid})")

Server script: /Users/josephazar/Desktop/WORK/GITHUB/agentic-ai-langchain-maf-workshop/utility/mcp-server.py
Exists: True


MCP server started (PID: 63247)


## Section 3: LangGraph + MCP (The Easy Way)

LangGraph has first-class MCP support through the `langchain-mcp-adapters` package. The integration is straightforward:

1. Open an SSE connection to the MCP server.
2. Create a `ClientSession` and initialize it.
3. Call `load_mcp_tools(session)` -- this returns a list of standard LangChain `Tool` objects.
4. Bind those tools to your LLM and build your graph as usual.

The adapter handles all the schema translation behind the scenes.

In [2]:
import json
from dotenv import load_dotenv
from typing import Annotated, Sequence, TypedDict

from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from langchain_openai import AzureChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from mcp import ClientSession
from mcp.client.sse import sse_client
from langchain_mcp_adapters.tools import load_mcp_tools

load_dotenv()

# ── Model ─────────────────────────────────────────────────────────────────────
model = AzureChatOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    model=os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1-mini"),
)

system_prompt = SystemMessage(
    "You are a helpful AI assistant. Use your tools to answer questions. "
    "If you call a tool, return the result in your response."
)

# ── Agent State ────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

print("LangGraph components initialized.")

LangGraph components initialized.


In [3]:
async def run_langgraph_mcp_agent(query: str):
    """Connect to the MCP server, load tools, build a LangGraph agent, and run a query."""

    async with sse_client("http://127.0.0.1:8787/sse") as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # --- This is the key line: one call loads all MCP tools ---
            tools = await load_mcp_tools(session)
            tool_map = {tool.name: tool for tool in tools}

            print(f"Discovered {len(tools)} tools: {[t.name for t in tools]}")

            # Bind tools to the model
            bound_model = model.bind_tools(tools=tools, tool_choice="auto")

            # ── Graph nodes ───────────────────────────────────────────────
            async def call_model(state: AgentState, config: RunnableConfig):
                messages = [system_prompt] + list(state["messages"])
                response = await bound_model.ainvoke(messages, config)
                return {"messages": [response]}

            async def tool_node(state: AgentState):
                outputs = []
                for tool_call in state["messages"][-1].tool_calls:
                    print(f"  Calling tool: {tool_call['name']}({tool_call['args']})")
                    result = await tool_map[tool_call["name"]].ainvoke(tool_call["args"])
                    outputs.append(
                        ToolMessage(
                            content=json.dumps(result),
                            name=tool_call["name"],
                            tool_call_id=tool_call["id"],
                        )
                    )
                return {"messages": outputs}

            def should_continue(state: AgentState):
                last = state["messages"][-1]
                return "continue" if last.tool_calls else "end"

            # ── Build graph ───────────────────────────────────────────────
            workflow = StateGraph(AgentState)
            workflow.add_node("agent", call_model)
            workflow.add_node("tools", tool_node)
            workflow.set_entry_point("agent")
            workflow.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
            workflow.add_edge("tools", "agent")
            graph = workflow.compile()

            # ── Run ───────────────────────────────────────────────────────
            print(f"\nQuery: {query}")
            result = await graph.ainvoke({"messages": [("user", query)]})
            answer = result["messages"][-1].content
            print(f"\nAnswer: {answer}")
            return answer

In [4]:
# Test the LangGraph + MCP agent
await run_langgraph_mcp_agent("What is the weather in New York?")

Discovered 2 tools: ['get_weather', 'get_current_time']

Query: What is the weather in New York?


  Calling tool: get_weather({'city': 'New York'})



Answer: The weather in New York is sunny with a temperature of 25 degrees Celsius (77 degrees Fahrenheit).


'The weather in New York is sunny with a temperature of 25 degrees Celsius (77 degrees Fahrenheit).'

**What just happened:**

1. The agent connected to the MCP server over SSE.
2. `load_mcp_tools()` called the server's `tools/list` endpoint, received JSON schemas for `get_weather` and `get_current_time`, and converted them into LangChain `Tool` objects.
3. The LLM saw both tools in its function-calling schema, chose `get_weather`, and the graph executed the call through the MCP session.
4. The tool result flowed back through the graph, and the LLM produced the final answer.

Total integration code: **one import and one function call** (`load_mcp_tools`). The adapter does the heavy lifting.

## Section 4: MAF + MCP (The Manual Way)

MAF (the agent framework from `agent_framework`) does not have a built-in MCP adapter. To connect it to an MCP server, we need to:

1. **Connect** to the server and list available tools.
2. **Read** each tool's JSON schema (name, description, input parameters).
3. **Build a Python function** with the correct signature using `inspect.Parameter`.
4. **Inject** `__name__`, `__doc__`, and `__signature__` so the framework can generate the correct LLM tool schema.

This is the most educational section -- it reveals what MCP adapters do under the hood.

### The Core Pattern: Dynamic Function Signature Construction

The challenge is that MCP tools are described as JSON schemas, but Python agent frameworks expect real Python functions with typed parameters. The bridge is `inspect.Parameter`:

```
MCP JSON Schema                    Python Function
+---------------------------+      +---------------------------+
| name: "get_weather"       | ---> | __name__ = "get_weather"  |
| description: "Retrieves" | ---> | __doc__ = "Retrieves..."  |
| properties:               |      | __signature__:            |
|   city:                   | ---> |   (city: str)             |
|     type: "string"       |      |                           |
+---------------------------+      +---------------------------+
```

In [5]:
import inspect
from mcp.types import TextContent, ImageContent, EmbeddedResource

# Map JSON schema types to Python types
TYPE_MAP = {
    "string": str,
    "integer": int,
    "number": float,
    "boolean": bool,
    "array": list,
    "object": dict,
}


def make_mcp_tool_func(session: ClientSession, tool):
    """
    Factory that takes an MCP tool descriptor and returns a Python async function
    with the correct signature, name, and docstring for use with agent frameworks.
    """
    tool_name = tool.name
    tool_description = tool.description or ""
    tool_schema = tool.inputSchema or {}
    properties = tool_schema.get("properties", {})
    required = tool_schema.get("required", [])

    # Step 1: Build inspect.Parameter objects from the JSON schema
    params = []
    for param_name, param_info in properties.items():
        json_type = param_info.get("type", "string")
        python_type = TYPE_MAP.get(json_type, str)
        default = inspect.Parameter.empty if param_name in required else None
        params.append(
            inspect.Parameter(
                name=param_name,
                kind=inspect.Parameter.POSITIONAL_OR_KEYWORD,
                default=default,
                annotation=python_type,
            )
        )

    # Step 2: Create the actual callable that invokes the MCP tool
    async def mcp_tool_func(*args, **kwargs):
        # Bind positional args to parameter names
        param_names = [p.name for p in params]
        for i, val in enumerate(args):
            if i < len(param_names):
                kwargs[param_names[i]] = val

        # Keep only the parameters the tool expects
        filtered_kwargs = {k: v for k, v in kwargs.items() if k in properties}

        print(f"  Calling MCP tool '{tool_name}' with: {filtered_kwargs}")

        try:
            result = await session.call_tool(tool_name, arguments=filtered_kwargs)

            if not result or not result.content:
                return f"Tool '{tool_name}' returned no content."

            parts = []
            for block in result.content:
                if isinstance(block, TextContent):
                    parts.append(block.text)
                elif isinstance(block, ImageContent):
                    parts.append(f"[Image: {block.mimeType}]")
                elif isinstance(block, EmbeddedResource):
                    parts.append(str(block.resource))
                elif hasattr(block, "text"):
                    parts.append(str(block.text))
                else:
                    parts.append(str(block))

            return "\n".join(parts) if parts else f"Tool '{tool_name}' returned empty content."

        except Exception as e:
            return f"Error calling tool '{tool_name}': {str(e)}"

    # Step 3: Inject metadata so the agent framework can build the LLM schema
    mcp_tool_func.__signature__ = inspect.Signature(params)
    mcp_tool_func.__name__ = tool_name
    mcp_tool_func.__doc__ = tool_description

    return mcp_tool_func


print("make_mcp_tool_func factory defined.")
print("\nThis function converts an MCP tool JSON schema into a Python async function")
print("with the correct __name__, __doc__, and __signature__ attributes.")

make_mcp_tool_func factory defined.

This function converts an MCP tool JSON schema into a Python async function
with the correct __name__, __doc__, and __signature__ attributes.


In [6]:
from agent_framework.openai import OpenAIChatClient


async def run_maf_mcp_agent(query: str):
    """Connect to MCP server, manually build tool functions, and run a MAF agent."""

    async with sse_client("http://127.0.0.1:8787/sse") as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # --- Manual tool discovery and construction ---
            mcp_tools = await session.list_tools()
            tools = [make_mcp_tool_func(session, tool) for tool in mcp_tools.tools]

            print(f"Discovered {len(tools)} tools: {[t.__name__ for t in tools]}")

            # Inspect the generated signatures
            for t in tools:
                print(f"  {t.__name__}{t.__signature__} -- {t.__doc__[:60]}...")

            # Build the MAF agent with the dynamically constructed tools
            agent = OpenAIChatClient().as_agent(
                name="MAF MCP Agent",
                description="An agent that uses tools loaded from an MCP server.",
                instructions=(
                    "You are a helpful assistant with access to tools. "
                    "If the user asks about weather, call get_weather. "
                    "If the user asks about time, call get_current_time. "
                    "Return the tool result in your response."
                ),
                tools=tools,
            )

            # Run the query
            print(f"\nQuery: {query}")
            response = await agent.run(query)
            print(f"\nAnswer: {response.text}")
            return response.text

In [7]:
# Test the MAF + MCP agent with the same query
await run_maf_mcp_agent("What is the weather in New York?")

Discovered 2 tools: ['get_weather', 'get_current_time']
  get_weather(city: str) -- Retrieves the current weather report for a specified city.

...
  get_current_time(city: str) -- Returns the current time in a specified city.

    Args:
   ...

Query: What is the weather in New York?


  Calling MCP tool 'get_weather' with: {'city': 'New York'}



Answer: The weather in New York is sunny with a temperature of 25 degrees Celsius (77 degrees Fahrenheit).


'The weather in New York is sunny with a temperature of 25 degrees Celsius (77 degrees Fahrenheit).'

**What the manual approach required:**

1. We called `session.list_tools()` to get tool descriptors from the MCP server.
2. For each tool, `make_mcp_tool_func` read the JSON schema and built `inspect.Parameter` objects with the correct names, types, and defaults.
3. It created a closure (`mcp_tool_func`) that calls `session.call_tool()` and parses the response.
4. It injected `__signature__`, `__name__`, and `__doc__` so that `OpenAIChatClient` could generate the correct function-calling schema for the LLM.

This is exactly what `langchain-mcp-adapters` does internally for LangGraph -- but here we had to write it ourselves.

## Section 5: Side-by-Side Comparison

Both agents connected to the same MCP server and answered the same question. The difference is entirely in how they **integrate** with MCP.

| Aspect | LangGraph | MAF |
|--------|-----------|-----|
| MCP adapter | `langchain-mcp-adapters` (built-in) | Manual signature injection |
| Lines of integration code | ~5 | ~40 |
| Tool discovery | `load_mcp_tools(session)` | `session.list_tools()` + factory function |
| Type mapping | Handled by adapter | Manual `TYPE_MAP` dictionary |
| Response parsing | Automatic | Manual (`TextContent`, `ImageContent`, etc.) |
| Maintenance burden | Adapter handles schema changes | Must update factory if MCP types change |
| Educational value | Lower (magic hidden) | Higher (shows what adapters do internally) |

### When to use each approach

- **Use the adapter** (LangGraph style) for production work. It is less code, fewer bugs, and maintained by the community.
- **Use the manual approach** (MAF style) when your framework lacks an MCP adapter, or when you need fine-grained control over how tool schemas are translated.

## Section 6: Cleanup and Summary

In [8]:
# Stop the MCP server
server_proc.terminate()
server_proc.wait()
print(f"MCP server (PID: {server_proc.pid}) terminated.")

MCP server (PID: 63247) terminated.


## Key Takeaways

1. **MCP separates tools from agents.** A tool server can be developed, deployed, and updated independently of the agents that consume it.
2. **FastMCP makes server development trivial.** Decorate a function with `@mcp.tool()` and the framework generates the JSON schema from the type hints and docstring.
3. **LangGraph integration is minimal.** With `langchain-mcp-adapters`, a single `load_mcp_tools()` call converts MCP tool descriptors into LangChain `Tool` objects.
4. **Manual integration reveals the mechanics.** Building Python function signatures from JSON schemas with `inspect.Parameter` shows exactly what adapter libraries do internally.
5. **The protocol is framework-agnostic.** Any agent framework can connect to an MCP server -- the only question is how much boilerplate is required.

---

## References

- `utility/mcp-server.py` -- The FastMCP server used in this notebook.
- `single-agents/langgraph/mcp_skills_langgraph.py` -- Full LangGraph + MCP agent with CLI loop.
- `single-agents/maf/mcp_skills_maf.py` -- Full MAF + MCP agent with CLI loop.

---

## Exercises

1. **Add a new tool to the MCP server.** Open `utility/mcp-server.py` and add a `get_exchange_rate(base: str, target: str) -> dict` tool. Restart the server and verify both agents discover the new tool automatically.

2. **Test with both agents.** After adding the new tool, run both the LangGraph and MAF agents against the updated server. Confirm that neither agent code needed to change.

3. **Add error handling for an unreachable server.** Wrap the `sse_client` connection in a try/except block and provide a helpful error message when the MCP server is not running.